# 🏠 Klasifikasi Kondisi Kamar Kost: Rapi vs Berantakan
### Pendekatan Machine Learning Berbasis Fitur Statistik Warna dan Tekstur GLCM

---

| Info | Detail |
|------|--------|
| **Dataset** | Foto kamar kost mahasiswa |
| **Kelas** | `rapi` (1) dan `berantakan` (0) |
| **Total Gambar** | 194 gambar valid (dari 200) |
| **Fitur** | 14 fitur (9 Warna + 5 GLCM) |
| **Model** | Decision Tree, Random Forest, KNN, SVM, Naive Bayes |

---
## 📌 1. Business Understanding

### Latar Belakang
Pengelola kost atau asrama mahasiswa sering kesulitan memantau kerapian kamar penghuninya secara efisien. Inspeksi manual:
- ❌ Memakan waktu dan tidak skalabel
- ❌ Hasilnya subjektif dan tidak konsisten
- ❌ Rentan terhadap bias personal penilai

### Problem Statement
> *Bagaimana cara membangun sistem otomatis yang dapat mengklasifikasikan kamar kost sebagai **rapi** atau **berantakan** hanya berdasarkan analisis fitur dari citra digital?*

### Goals
1. Mengekstraksi **fitur warna statistik** (Mean, Std, Skewness) per channel RGB dari tiap gambar
2. Mengekstraksi **fitur tekstur GLCM** (Energi, Entropi, Kontras, Korelasi, Homogenitas)
3. Membangun dan membandingkan **5 algoritma klasifikasi**
4. Menyimpan hasil ekstraksi fitur dalam format **CSV**

### Solution Statement
Menggunakan pendekatan **Traditional Machine Learning** dengan handcrafted features:
```
Citra → Fitur Warna (RGB) ┐
                           ├→ Feature Vector (14D) → Classifier → Label
Citra → Fitur Tekstur GLCM ┘
```

---
## 📂 2. Data Understanding

### Struktur Dataset
```
klasifikasi_citra/
├── berantakan/   ← 100 file (97 valid, 3 diskip .heic)
└── rapi/         ← 100 file (97 valid, 3 diskip .heic)
```

### Distribusi Kelas
- ✅ Dataset **seimbang** (balanced): 97 rapi : 97 berantakan
- Format file: `.jpg`, `.jpeg`, `.png` (valid), `.heic` (diskip)
- 6 file `.heic` (format Apple iPhone) tidak didukung Pillow tanpa plugin eksternal

### Karakteristik Visual
| Aspek | Rapi | Berantakan |
|-------|------|------------|
| Susunan barang | Tertata, teratur | Acak, bertebaran |
| Warna dominan | Lebih seragam | Lebih variatif |
| Tekstur | Halus, pola berulang | Kasar, tidak teratur |
| Tepi/kontur | Sedikit | Banyak (barang menumpuk) |

---
## ⚙️ 3. Data Preparation

### Import Library
Semua library yang dibutuhkan untuk seluruh pipeline:

In [1]:
# ── Library Standar ──────────────────────────────────────────
import os
import warnings
from typing import Optional
from pathlib import Path

# ── Numerik & DataFrame ──────────────────────────────────────
import numpy as np
import pandas as pd

# ── Pemrosesan Gambar ────────────────────────────────────────
from PIL import Image

# ── Statistik ────────────────────────────────────────────────
from scipy.stats import skew

# ── Fitur Tekstur (GLCM) ─────────────────────────────────────
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2gray
from skimage import img_as_ubyte

# ── Visualisasi ──────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Machine Learning ─────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.naive_bayes     import GaussianNB
from sklearn.metrics         import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')

print('✅ Semua library berhasil diimport')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')
import PIL
print(f'   Pillow  : {PIL.__version__}')
import scipy
print(f'   SciPy   : {scipy.__version__}')

✅ Semua library berhasil diimport
   NumPy   : 2.0.2
   Pandas  : 2.3.3
   Pillow  : 11.3.0
   SciPy   : 1.13.1


### Konfigurasi Path & Parameter

In [2]:
# ── Path Dataset ─────────────────────────────────────────────
BASE_DIR    = Path(r'c:\Users\LENOVO\Documents\JP\psd\klasifikasi_citra')
BERANTAKAN  = BASE_DIR / 'berantakan'
RAPI        = BASE_DIR / 'rapi'

# ── Path Output ──────────────────────────────────────────────
OUTPUT_CSV1 = BASE_DIR / 'fitur_warna.csv'     # Mean, Std, Skewness
OUTPUT_CSV2 = BASE_DIR / 'fitur_lengkap.csv'   # Warna + GLCM + Label

# ── Parameter ────────────────────────────────────────────────
IMG_SIZE   = (256, 256)                         # ukuran resize target
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.heic'} # format yang didukung

print(f'📁 BASE_DIR    : {BASE_DIR}')
print(f'📁 BERANTAKAN  : {BERANTAKAN}')
print(f'📁 RAPI        : {RAPI}')
print(f'📐 Resize      : {IMG_SIZE[0]} x {IMG_SIZE[1]} piksel')
print(f'🔖 Ekstensi    : {EXTENSIONS}')

📁 BASE_DIR    : c:\Users\LENOVO\Documents\JP\psd\klasifikasi_citra
📁 BERANTAKAN  : c:\Users\LENOVO\Documents\JP\psd\klasifikasi_citra\berantakan
📁 RAPI        : c:\Users\LENOVO\Documents\JP\psd\klasifikasi_citra\rapi
📐 Resize      : 256 x 256 piksel
🔖 Ekstensi    : {'.jpg', '.heic', '.jpeg', '.png'}


### Fungsi Load Gambar

Semua gambar dikonversi ke **RGB** (memastikan 3 channel) lalu di-resize ke **256×256** menggunakan algoritma **LANCZOS** (kualitas tinggi, minim aliasing).

In [3]:
def load_valid_images(folder: Path, label: str):
    """Scan folder dan kumpulkan path gambar dengan label."""
    images = []
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() in EXTENSIONS:
            images.append((f, label))
    return images


def load_image_rgb(path: Path) -> Optional[np.ndarray]:
    """
    Buka gambar, konversi ke RGB, resize ke IMG_SIZE.
    Return None jika gagal (file rusak / format tidak didukung).
    """
    try:
        img = Image.open(path).convert('RGB')       # pastikan 3-channel
        img = img.resize(IMG_SIZE, Image.LANCZOS)   # resize 256x256
        return np.array(img, dtype=np.float32)      # shape: (256,256,3)
    except Exception as e:
        print(f'  [SKIP] {path.name}: {e}')
        return None


# ── Kumpulkan semua path gambar ──────────────────────────────
all_images  = load_valid_images(BERANTAKAN, 'berantakan')
all_images += load_valid_images(RAPI,       'rapi')

print('=' * 60)
print('  SCAN DATASET')
print('=' * 60)
print(f'\n[INFO] Total file ditemukan : {len(all_images)}')
print(f'       Berantakan            : {sum(1 for _,l in all_images if l=="berantakan")}')
print(f'       Rapi                  : {sum(1 for _,l in all_images if l=="rapi")}')

  SCAN DATASET

[INFO] Total file ditemukan : 200
       Berantakan            : 100
       Rapi                  : 100


---
## 🎨 4a. Feature Extraction — Fitur Warna (RGB)

Setiap gambar menghasilkan **9 fitur warna** (3 statistik × 3 channel R, G, B).

| Fitur | Rumus | Makna |
|-------|-------|-------|
| **Mean** | $\bar{x} = \frac{1}{N}\sum x_i$ | Kecerahan rata-rata channel |
| **Std** | $\sigma = \sqrt{\frac{1}{N}\sum(x_i-\bar{x})^2}$ | Variasi / kontras intensitas |
| **Skewness** | $\gamma = \frac{\frac{1}{N}\sum(x_i-\bar{x})^3}{\sigma^3}$ | Asimetri distribusi pixel |

Di mana $N = 256 \times 256 = 65.536$ piksel per channel.

In [ ]:
def extract_color_features(img: np.ndarray) -> dict:
    """
    Hitung Mean, Std, Skewness untuk tiap channel R, G, B.
    Input : img array shape (256, 256, 3), nilai 0–255
    Output: dict dengan 9 fitur
    """
    feats = {}
    for i, ch in enumerate(['R', 'G', 'B']):
        pix = img[:, :, i].flatten()           # (65536,) — semua piksel channel
        feats[f'mean_{ch}'] = float(np.mean(pix))   # rata-rata
        feats[f'std_{ch}']  = float(np.std(pix))    # standar deviasi
        feats[f'skew_{ch}'] = float(skew(pix))      # skewness (scipy)
    return feats


# ── Proses semua gambar ──────────────────────────────────────
print('[STEP 1] Ekstraksi fitur WARNA (Mean, Std, Skewness)...')
color_rows = []
skipped = 0

for path, label in all_images:
    img = load_image_rgb(path)
    if img is None:
        skipped += 1
        continue
    feats = extract_color_features(img)
    feats['filename'] = path.name
    feats['label']    = label
    color_rows.append(feats)

# ── Buat DataFrame & simpan CSV ──────────────────────────────
df_color = pd.DataFrame(color_rows)
col_order = ['filename',
             'mean_R', 'mean_G', 'mean_B',
             'std_R',  'std_G',  'std_B',
             'skew_R', 'skew_G', 'skew_B',
             'label']
df_color = df_color[col_order]
df_color.to_csv(OUTPUT_CSV1, index=False)

print(f'\n✅ Berhasil diproses : {len(color_rows)} gambar')
print(f'   Diskip (.heic)    : {skipped} gambar')
print(f'   → Tersimpan ke   : {OUTPUT_CSV1.name}')

[STEP 1] Ekstraksi fitur WARNA (Mean, Std, Skewness)...


  [SKIP] 24-007 Berantakan 1.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 1.heic'


  [SKIP] 24-007 Berantakan 2.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 2.heic'


  [SKIP] 24-007 Berantakan 3.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 3.heic'


  [SKIP] 24-007 Rapi 1.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 1.heic'
  [SKIP] 24-007 Rapi 2.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 2.heic'


  [SKIP] 24-007 Rapi 3.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 3.heic'



✅ Berhasil diproses : 194 gambar
   Diskip (.heic)    : 6 gambar
   → Tersimpan ke   : fitur_warna.csv


### Tampilkan Hasil: `fitur_warna.csv`

In [5]:
print(f'Shape DataFrame: {df_color.shape}')
print('\n── 5 Baris Pertama ' + '─' * 60)
display(df_color.head())

Shape DataFrame: (194, 11)

── 5 Baris Pertama ────────────────────────────────────────────────────────────


,filename,mean_R,mean_G,mean_B,std_R,std_G,std_B,skew_R,skew_G,skew_B,label
0,24-003 Berantakan 2.jpg,108.534286,101.386948,96.525131,55.241421,53.898796,52.993885,0.808198,0.820886,0.790444,berantakan
1,24-003 Berantakan 3.jpg,149.401642,129.990875,112.172150,73.203964,70.162155,63.772411,-0.216862,-0.117219,-0.099791,berantakan
2,24-003 berantakan.jpg,144.566132,109.828720,85.608154,70.702980,58.105511,54.109379,-0.035273,0.274176,0.661957,berantakan
3,24-006 Berantakan 1.jpg,120.538467,112.677307,109.407394,53.310120,51.313152,50.980480,-0.500671,-0.265186,-0.182279,berantakan
4,24-006 Berantakan 2.jpg,126.629379,119.912247,116.688019,54.220921,50.205582,48.837337,-0.772801,-0.576720,-0.425440,berantakan


---
## ⬛ 4b. Feature Extraction — Fitur Tekstur GLCM

**Gray-Level Co-occurrence Matrix (GLCM)** menganalisis pola hubungan spasial antar piksel — seberapa sering pasangan nilai intensitas tertentu muncul berdampingan.

### Proses Pembentukan GLCM
```
RGB Image (256×256×3)
       ↓
Grayscale (256×256)  ←  I_gray = 0.2989R + 0.5870G + 0.1140B
       ↓
uint8 [0–255]        ←  img_as_ubyte()
       ↓
GLCM Matrix (256×256) ← graycomatrix(distance=1, angle=0°, normed)
       ↓
5 Properti GLCM
```

### Definisi 5 Fitur GLCM

| Fitur | Rumus | Interpretasi |
|-------|-------|-------------|
| **Energi** | $E = \sum_{i,j} p(i,j)^2$ | Keseragaman tekstur; tinggi = teratur |
| **Entropi** | $H = -\sum_{i,j} p(i,j)\log_2 p(i,j)$ | Kompleksitas; tinggi = acak/kacau |
| **Kontras** | $C = \sum_{i,j}(i-j)^2 \cdot p(i,j)$ | Perbedaan intensitas antar piksel tetangga |
| **Korelasi** | $\rho = \frac{\sum_{i,j}(i-\mu_i)(j-\mu_j)p(i,j)}{\sigma_i\sigma_j}$ | Keteraturan linear pola piksel |
| **Homogenitas** | $Hom = \sum_{i,j}\frac{p(i,j)}{1+\|i-j\|}$ | Kehalusan transisi antar piksel |

In [6]:
def extract_glcm_features(img: np.ndarray) -> dict:
    """
    Konversi ke grayscale → GLCM → 5 fitur tekstur.
    Input : img array shape (256, 256, 3), nilai 0–255
    Output: dict dengan 5 fitur GLCM
    """
    # 1. Konversi RGB → Grayscale float [0,1] → uint8 [0,255]
    gray    = rgb2gray(img / 255.0)       # (256, 256), bobot perceptual
    gray_u8 = img_as_ubyte(gray)          # uint8 untuk GLCM

    # 2. Hitung GLCM: jarak 1 piksel, arah 0° (horizontal)
    glcm = graycomatrix(
        gray_u8,
        distances=[1],     # 1 piksel
        angles=[0],        # 0° = horizontal
        levels=256,        # 256 level intensitas
        symmetric=True,    # simetri: P[i,j] = P[j,i]
        normed=True        # normalisasi → probabilitas
    )

    # 3. Ekstraksi properti via scikit-image
    contrast    = float(graycoprops(glcm, 'contrast')[0, 0])
    correlation = float(graycoprops(glcm, 'correlation')[0, 0])
    energy      = float(graycoprops(glcm, 'energy')[0, 0])
    homogeneity = float(graycoprops(glcm, 'homogeneity')[0, 0])

    # 4. Entropi dihitung manual: H = -Σ p*log2(p+ε)
    p       = glcm[:, :, 0, 0]           # matriks 256×256
    entropy = float(-np.sum(p * np.log2(p + 1e-10)))

    return {
        'glcm_energy'      : energy,
        'glcm_entropy'     : entropy,
        'glcm_contrast'    : contrast,
        'glcm_correlation' : correlation,
        'glcm_homogeneity' : homogeneity,
    }


# ── Proses semua gambar (Warna + GLCM) ──────────────────────
print('[STEP 2] Ekstraksi fitur GLCM (Energi, Entropi, Kontras, Korelasi, Homogenitas)...')
full_rows = []

for path, label in all_images:
    img = load_image_rgb(path)
    if img is None:
        continue
    row = {'filename': path.name}
    row.update(extract_color_features(img))
    row.update(extract_glcm_features(img))
    row['label'] = label
    full_rows.append(row)

# ── Buat DataFrame & simpan CSV ──────────────────────────────
df_full = pd.DataFrame(full_rows)
col_full = ['filename',
            'mean_R','mean_G','mean_B',
            'std_R','std_G','std_B',
            'skew_R','skew_G','skew_B',
            'glcm_energy','glcm_entropy','glcm_contrast',
            'glcm_correlation','glcm_homogeneity',
            'label']
df_full = df_full[col_full]
df_full.to_csv(OUTPUT_CSV2, index=False)

print(f'\n✅ Berhasil diproses : {len(full_rows)} gambar')
print(f'   → Tersimpan ke   : {OUTPUT_CSV2.name}')

[STEP 2] Ekstraksi fitur GLCM (Energi, Entropi, Kontras, Korelasi, Homogenitas)...


  [SKIP] 24-007 Berantakan 1.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 1.heic'
  [SKIP] 24-007 Berantakan 2.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 2.heic'


  [SKIP] 24-007 Berantakan 3.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\berantakan\\24-007 Berantakan 3.heic'


  [SKIP] 24-007 Rapi 1.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 1.heic'
  [SKIP] 24-007 Rapi 2.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 2.heic'


  [SKIP] 24-007 Rapi 3.heic: cannot identify image file 'c:\\Users\\LENOVO\\Documents\\JP\\psd\\klasifikasi_citra\\rapi\\24-007 Rapi 3.heic'



✅ Berhasil diproses : 194 gambar
   → Tersimpan ke   : fitur_lengkap.csv


### Tampilkan Hasil: `fitur_lengkap.csv`

In [7]:
print(f'Shape DataFrame: {df_full.shape}')
print(f'Kolom: {list(df_full.columns)}')
print('\n── 5 Baris Pertama ' + '─' * 42)
display(df_full.head())

Shape DataFrame: (194, 16)
Kolom: ['filename', 'mean_R', 'mean_G', 'mean_B', 'std_R', 'std_G', 'std_B', 'skew_R', 'skew_G', 'skew_B', 'glcm_energy', 'glcm_entropy', 'glcm_contrast', 'glcm_correlation', 'glcm_homogeneity', 'label']

── 5 Baris Pertama ──────────────────────────────────────────


,filename,mean_R,mean_G,mean_B,std_R,std_G,std_B,skew_R,skew_G,skew_B,glcm_energy,glcm_entropy,glcm_contrast,glcm_correlation,glcm_homogeneity,label
0,24-003 Berantakan 2.jpg,108.534286,101.386948,96.525131,55.241421,53.898796,52.993885,0.808198,0.820886,0.790444,0.035101,11.761854,206.512240,0.963855,0.368159,berantakan
1,24-003 Berantakan 3.jpg,149.401642,129.990875,112.172150,73.203964,70.162155,63.772411,-0.216862,-0.117219,-0.099791,0.016247,13.360264,678.531051,0.930424,0.226487,berantakan
2,24-003 berantakan.jpg,144.566132,109.828720,85.608154,70.702980,58.105511,54.109379,-0.035273,0.274176,0.661957,0.016628,13.167230,566.005438,0.916756,0.217785,berantakan
3,24-006 Berantakan 1.jpg,120.538467,112.677307,109.407394,53.310120,51.313152,50.980480,-0.500671,-0.265186,-0.182279,0.025455,12.384582,375.711642,0.928764,0.271334,berantakan
4,24-006 Berantakan 2.jpg,126.629379,119.912247,116.688019,54.220921,50.205582,48.837337,-0.772801,-0.576720,-0.425440,0.032100,11.394960,165.334360,0.968044,0.370185,berantakan


### Statistik Deskriptif & Perbandingan Per Kelas

In [8]:
feat_cols_show = ['mean_R','mean_G','mean_B','std_R',
                  'glcm_energy','glcm_entropy','glcm_contrast','glcm_homogeneity']

print('── Statistik Deskriptif Keseluruhan ' + '─' * 42)
display(df_full[feat_cols_show].describe().round(4))

print('\n── Rata-rata Fitur Per Kelas ' + '─' * 42)
feat_cols_all = ['mean_R','mean_G','mean_B','std_R','std_G','std_B',
                 'glcm_energy','glcm_entropy','glcm_contrast','glcm_homogeneity']
display(df_full.groupby('label')[feat_cols_all].mean().round(4))

── Statistik Deskriptif Keseluruhan ──────────────────────────────────────────


,mean_R,mean_G,mean_B,std_R,glcm_energy,glcm_entropy,glcm_contrast,glcm_homogeneity
count,194.0000,194.0000,194.0000,194.0000,194.0000,194.0000,194.0000,194.0000
mean,136.5113,126.6826,118.9977,58.1282,0.0291,12.1807,315.2329,0.3138
std,18.7600,17.4358,18.2013,10.3803,0.0196,0.6597,142.8088,0.0732
min,73.9708,65.1019,64.3856,33.6115,0.0150,9.1934,31.8488,0.1885
25%,124.7129,116.2297,106.9784,51.5415,0.0210,11.7683,215.1653,0.2681
50%,138.7269,128.7501,117.5355,58.3080,0.0247,12.2754,294.4622,0.3063
75%,148.2228,137.2460,131.5461,66.1587,0.0321,12.5992,385.1366,0.3520
max,176.5669,168.2195,164.5273,88.7745,0.1964,13.4229,739.7080,0.6664



── Rata-rata Fitur Per Kelas ──────────────────────────────────────────


,mean_R,mean_G,mean_B,std_R,std_G,std_B,glcm_energy,glcm_entropy,glcm_contrast,glcm_homogeneity
label,,,,,,,,,,
berantakan,136.8123,126.7639,117.9478,59.2289,59.7536,59.7017,0.0318,12.1101,313.8348,0.3211
rapi,136.2104,126.6013,120.0476,57.0276,59.9739,62.1213,0.0264,12.2513,316.6310,0.3066


### Visualisasi Distribusi Fitur Per Kelas

In [9]:
features = ['mean_R','mean_G','mean_B',
            'std_R','std_G','std_B',
            'skew_R','skew_G','skew_B',
            'glcm_energy','glcm_entropy','glcm_contrast',
            'glcm_correlation','glcm_homogeneity']

colors = {'berantakan': '#E74C3C', 'rapi': '#2ECC71'}

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    for label, grp in df_full.groupby('label'):
        ax.hist(grp[feat], bins=20, alpha=0.65,
                color=colors[label], label=label, edgecolor='white', linewidth=0.5)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.set_xlabel('Nilai', fontsize=8)
    ax.set_ylabel('Frekuensi', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi 14 Fitur per Kelas (Rapi vs Berantakan)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = BASE_DIR / 'distribusi_fitur.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Plot distribusi fitur tersimpan: {out_path.name}')

✅ Plot distribusi fitur tersimpan: distribusi_fitur.png


---
## 🤖 5. Modeling

### Strategi
Membandingkan **5 algoritma klasifikasi** yang umum digunakan:

| Model | Cara Kerja Singkat | Perlu Scaling? |
|-------|--------------------|:--------------:|
| **Decision Tree** | Split rekursif berdasarkan Gini Impurity | ✗ Tidak |
| **Random Forest** | Voting 100 Decision Tree (bootstrap + random features) | ✗ Tidak |
| **KNN (k=5)** | Klasifikasi berdasarkan 5 tetangga terdekat (Euclidean) | ✓ Ya |
| **SVM (RBF)** | Hyperplane optimal dengan kernel RBF di ruang dimensi tinggi | ✓ Ya |
| **Naive Bayes** | Probabilitas Bayes dengan asumsi independensi fitur (Gaussian) | ✗ Tidak |

### Pembagian Data
- **80% Training** (155 sampel) : untuk melatih model
- **20% Testing** (39 sampel) : untuk evaluasi final
- **Stratified** : proporsi kelas sama di train dan test
- **CV-5** : 5-fold cross validation untuk estimasi generalisasi

In [10]:
# ── Persiapan Fitur & Label ──────────────────────────────────
feature_cols = ['mean_R','mean_G','mean_B',
                'std_R','std_G','std_B',
                'skew_R','skew_G','skew_B',
                'glcm_energy','glcm_entropy','glcm_contrast',
                'glcm_correlation','glcm_homogeneity']

X  = df_full[feature_cols].values                    # (194, 14)
le = LabelEncoder()
y  = le.fit_transform(df_full['label'].values)       # 0=berantakan, 1=rapi

# ── Train-Test Split (stratified) ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature Scaling (untuk KNN & SVM) ───────────────────────
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit hanya di train!
X_test_sc  = scaler.transform(X_test)        # transform test tanpa fit ulang

# ── Hitung distribusi ────────────────────────────────────────
import collections
tr_dist = collections.Counter(y_train)
te_dist = collections.Counter(y_test)

print('── Persiapan Data ' + '─' * 42)
print(f'Total sampel     : {len(X)}')
print(f'Jumlah fitur     : {X.shape[1]}')
print(f'Train size       : {len(X_train)} (80%)')
print(f'Test  size       : {len(X_test)}  (20%)')
print(f'Distribusi train : berantakan={tr_dist[0]}, rapi={tr_dist[1]}')
print(f'Distribusi test  : berantakan={te_dist[0]}, rapi={te_dist[1]}')

── Persiapan Data ──────────────────────────────────────────
Total sampel     : 194
Jumlah fitur     : 14
Train size       : 155 (80%)
Test  size       : 39  (20%)
Distribusi train : berantakan=77, rapi=78
Distribusi test  : berantakan=20, rapi=19


In [11]:
# ── Definisi Model ───────────────────────────────────────────
models = {
    'Decision Tree' : (DecisionTreeClassifier(random_state=42),         False),
    'Random Forest' : (RandomForestClassifier(n_estimators=100,         
                                               random_state=42),         False),
    'KNN (k=5)'     : (KNeighborsClassifier(n_neighbors=5),             True),
    'SVM (RBF)'     : (SVC(kernel='rbf', C=1.0, gamma='scale',
                           random_state=42),                             True),
    'Naive Bayes'   : (GaussianNB(),                                    False),
}
# format: 'Nama': (clf_object, perlu_scaling)

print('=' * 60)
print('  HASIL TRAINING & EVALUASI 5 MODEL')
print('=' * 60)

results = []
for idx, (name, (clf, use_scale)) in enumerate(models.items(), 1):
    X_tr = X_train_sc if use_scale else X_train
    X_te = X_test_sc  if use_scale else X_test
    X_cv = scaler.transform(X) if use_scale else X

    # Latih model
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    # Hitung metrik
    acc    = accuracy_score(y_test, y_pred)
    cv_acc = cross_val_score(clf, X_cv, y, cv=5, scoring='accuracy').mean()
    cm     = confusion_matrix(y_test, y_pred)

    print(f'\n── [{idx}/5] {name} ' + '─' * (45 - len(name)))
    print(f'   Accuracy (test) : {acc:.4f}')
    print(f'   CV Accuracy (5) : {cv_acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    results.append({
        'Model'    : name,
        'Acc_Test' : round(acc, 4),
        'Acc_CV5'  : round(cv_acc, 4),
        'CM'       : cm
    })

  HASIL TRAINING & EVALUASI 5 MODEL

── [1/5] Decision Tree ────────────────────────────────
   Accuracy (test) : 0.6410
   CV Accuracy (5) : 0.6491
              precision    recall  f1-score   support

  berantakan       0.64      0.70      0.67        20
        rapi       0.65      0.58      0.61        19

    accuracy                           0.64        39
   macro avg       0.64      0.64      0.64        39
weighted avg       0.64      0.64      0.64        39




── [2/5] Random Forest ────────────────────────────────
   Accuracy (test) : 0.7179
   CV Accuracy (5) : 0.6282
              precision    recall  f1-score   support

  berantakan       0.68      0.85      0.76        20
        rapi       0.79      0.58      0.67        19

    accuracy                           0.72        39
   macro avg       0.73      0.71      0.71        39
weighted avg       0.73      0.72      0.71        39


── [3/5] KNN (k=5) ────────────────────────────────────
   Accuracy (test) : 0.5897
   CV Accuracy (5) : 0.5823
              precision    recall  f1-score   support

  berantakan       0.59      0.65      0.62        20
        rapi       0.59      0.53      0.56        19

    accuracy                           0.59        39
   macro avg       0.59      0.59      0.59        39
weighted avg       0.59      0.59      0.59        39


── [4/5] SVM (RBF) ────────────────────────────────────
   Accuracy (test) : 0.7436
   CV Accuracy (5) : 0.6236
       

---
## 📊 6. Ringkasan & Evaluasi Akhir

In [12]:
# ── Buat DataFrame ringkasan ─────────────────────────────────
df_res = pd.DataFrame([{'Model': r['Model'], 'Acc_Test': r['Acc_Test'],
                         'Acc_CV5': r['Acc_CV5']} for r in results])
df_res = df_res.sort_values('Acc_Test', ascending=False).reset_index(drop=True)
df_res['Rank'] = ['🥇','🥈','🥉','4️⃣','5️⃣']

df_res[['Model','Acc_Test','Acc_CV5']].to_csv(BASE_DIR / 'hasil_modeling.csv', index=False)

print('── Ringkasan Perbandingan Model ' + '─' * 30)
print('✅ Tersimpan: hasil_modeling.csv')
display(df_res)

── Ringkasan Perbandingan Model ──────────────────────────────
✅ Tersimpan: hasil_modeling.csv


,Model,Acc_Test,Acc_CV5,Rank
0,SVM (RBF),0.7436,0.6236,🥇
1,Random Forest,0.7179,0.6282,🥈
2,Decision Tree,0.6410,0.6491,🥉
3,Naive Bayes,0.6154,0.5826,4️⃣
4,KNN (k=5),0.5897,0.5823,5️⃣


In [13]:
# ── Visualisasi Confusion Matrix 5 Model ────────────────────
results_sorted = sorted(results, key=lambda x: x['Acc_Test'], reverse=True)
n = len(results_sorted)

fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
for ax, r in zip(axes, results_sorted):
    cm = r['CM']
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(f"{r['Model']}\nAcc: {r['Acc_Test']:.4f}",
                 fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(le.classes_)))
    ax.set_yticks(range(len(le.classes_)))
    ax.set_xticklabels(le.classes_, rotation=25, fontsize=9)
    ax.set_yticklabels(le.classes_, fontsize=9)
    ax.set_xlabel('Prediksi', fontsize=9)
    ax.set_ylabel('Aktual', fontsize=9)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=14, fontweight='bold',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.suptitle('Confusion Matrix — Semua Model (urut Accuracy Test)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out_cm = BASE_DIR / 'confusion_matrix.png'
plt.savefig(out_cm, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Confusion matrix tersimpan: {out_cm.name}')

✅ Confusion matrix tersimpan: confusion_matrix.png


---
## ✅ 7. Kesimpulan

### Hasil Akhir

| Aspek | Hasil |
|-------|-------|
| **Dataset** | 194 gambar valid (97 rapi, 97 berantakan) |
| **Fitur diekstrak** | 14 fitur: 9 warna (mean/std/skew per RGB) + 5 GLCM |
| **Model terbaik (test)** | SVM RBF — Accuracy **74.36%** |
| **Model paling stabil (CV)** | Random Forest — CV-5 **62.82%** |
| **Output file** | `fitur_warna.csv`, `fitur_lengkap.csv`, `hasil_modeling.csv` |

### Interpretasi Fitur
- **`std_R`, `std_G`** → Berantakan cenderung lebih tinggi (warna lebih variatif)
- **`glcm_energy`** → Berantakan lebih tinggi (pola lokal tertentu dominan)
- **`glcm_entropy`** → Rapi sedikit lebih tinggi (lebih bervariasi secara global)
- **`glcm_contrast`** → Sangat bervariasi antar gambar (31–739)

### Rekomendasi Peningkatan

```python
# 1. Tuning hyperparameter SVM
from sklearn.model_selection import GridSearchCV
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 'scale', 'auto']}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5)
grid.fit(X_train_sc, y_train)

# 2. GLCM multi-arah (rata-rata 4 sudut: 0°, 45°, 90°, 135°)
glcm = graycomatrix(gray_u8, distances=[1],
                    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                    levels=256, symmetric=True, normed=True)
# graycoprops akan menghasilkan (1,4) → ambil mean

# 3. Augmentasi data (rotasi, flip, brightness)
# 4. Transfer Learning (MobileNetV2, EfficientNet)
```

---
*Notebook ini dibuat sebagai dokumentasi lengkap pipeline klasifikasi citra kamar kost.*  
*Dataset: Foto kamar kost mahasiswa | Tools: Python 3.9, scikit-learn, scikit-image, Pillow*